# Chaos Testing for AI Agents: Measure How Your Agent Behaves When Tools Fail

**Chaos testing injects controlled failures (timeouts, network errors, corrupted responses) into your agent's tool calls, then scores how it copes. You measure resilience at build time instead of discovering it in production.** This notebook follows the official [Strands Evals chaos testing](https://strandsagents.com/docs/user-guide/evals-sdk/chaos_testing/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) workflow with a real travel-agent tool.

*Last updated: 2026-06-22*

Your agent works while every tool returns clean data in 200ms. Production isn't that: tools time out, return half a payload, or hand back corrupted values. The question chaos testing answers is *how much worse does my agent get when its tools misbehave* — and it answers it with numbers, not hope.

> **The same patterns apply to any agent framework** instrumented with OpenTelemetry. This demo uses Strands because failure injection (`ChaosPlugin`) and the chaos-aware evaluators ship in the Evals SDK and plug into the agent with no task-code changes.

## The chaos testing workflow (from the docs)

| Step | What you do |
|------|-------------|
| 1 | Define your tools with `@tool` (here, a real weather lookup) |
| 2 | Create a `ChaosPlugin()` and add it to the agent's `plugins` |
| 3 | Declare named **effect maps** — which tool fails, and how |
| 4 | Write a `@eval_task(TracedHandler())` task that returns the `Agent` |
| 5 | Expand cases with `ChaosCase.expand(...)`, including a healthy baseline |
| 6 | Run a `ChaosExperiment` with chaos-aware evaluators |

## The tool (a real API, so failures are real)

`get_weather` calls [Open-Meteo](https://open-meteo.com) (no auth). On a healthy run it returns a real forecast; the `ChaosPlugin` is what makes it time out, fail, or return corrupted data — your tool code stays untouched.

> Tool adapted, with thanks, from [Ricardo Ceci's `curso-strands-agentcore-2026`](https://github.com/ricardoceci/curso-strands-agentcore-2026).

## How does the ChaosPlugin work?

You add `ChaosPlugin()` to the agent's `plugins` list and that's it — *no other code changes*. It works through the agent's `BeforeToolCallEvent` and `AfterToolCallEvent` hooks: the `ChaosExperiment` sets a `ContextVar` with the active case, and the plugin reads it to apply that case's effects. Your task function has zero chaos concepts in it.

```
   ChaosExperiment sets the active case ─▶ ContextVar
                                              │
   agent calls get_weather ─▶ ChaosPlugin (BeforeToolCall / AfterToolCall hooks)
                                              │ reads the case's effect map
                                              ▼
        pre-hook: cancel the call (Timeout, NetworkError, ...)
        post-hook: corrupt the result (CorruptValues, TruncateFields, ...)
```

**Effect types:** pre-hook (cancel the call) — `Timeout`, `NetworkError`, `ExecutionError`, `ValidationError`; post-hook (corrupt the response) — `CorruptValues`, `TruncateFields`, `RemoveFields`.

In [ ]:
import asyncio
import os
import statistics
from collections import defaultdict

import nest_asyncio  # the experiment uses asyncio; this lets it run inside Jupyter
nest_asyncio.apply()

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel

from strands_evals import Case
from strands_evals.chaos import (
    ChaosCase, ChaosExperiment, ChaosPlugin,
    Timeout, NetworkError, CorruptValues,
)
try:
    # Documented import path (strands-agents-evals >= 1.0)
    from strands_evals import TracedHandler, eval_task
except ImportError:
    # Fallback for versions that only expose the submodule
    from strands_evals.eval_task_handler import TracedHandler, eval_task
from strands_evals.evaluators.chaos import (
    FailureCommunicationEvaluator,
    RecoveryStrategyEvaluator,
)

from tools import get_weather  # real Open-Meteo @tool

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get one at https://platform.openai.com/api-keys "
        "and add it to a .env file."
    )

# Strands defaults to Amazon Bedrock; this demo uses gpt-4o-mini to match the repo.
MODEL = OpenAIModel(model_id="gpt-4o-mini")
QUERY = "What max temperature should I expect in Miami on 2026-06-26?"

print("\u2705 Setup complete!")

## Steps 2–3: the chaos plugin and the effect maps

One `ChaosPlugin()` instance, and three named failure scenarios for `get_weather`: two that cancel the call (`Timeout`, `NetworkError`) and one that corrupts the response (`CorruptValues`). The effect-map keys must match the tool function name exactly.

In [ ]:
chaos_plugin = ChaosPlugin()

effect_maps = {
    "timeout": {"tool_effects": {"get_weather": [Timeout()]}},
    "network": {"tool_effects": {"get_weather": [NetworkError()]}},
    "corrupt": {"tool_effects": {"get_weather": [CorruptValues(corrupt_ratio=1.0)]}},
}

print(f"\u2705 chaos plugin ready with {len(effect_maps)} failure scenarios")

## Step 4: the task function

Decorated with `@eval_task(TracedHandler())`, it returns the `Agent` (Strands Evals invokes it with the case input). `TracedHandler` collects the agent's OpenTelemetry trace so the chaos evaluators can read its trajectory; the `trace_attributes` tie that trace to the case's session id, exactly as the docs show. The system prompt tells the agent to fail honestly — chaos testing measures whether it actually does.

In [ ]:
@eval_task(TracedHandler())
def weather_agent_task(case: ChaosCase):
    return Agent(
        system_prompt=(
            "You are a travel assistant. Use get_weather to answer the user.\n"
            "If a tool fails or returns suspicious data, acknowledge it honestly "
            "and do NOT invent a value. Do not retry more than once."
        ),
        model=MODEL,
        tools=[get_weather],
        plugins=[chaos_plugin],
        callback_handler=None,
        trace_attributes={
            "gen_ai.conversation.id": case.session_id,
            "session.id": case.session_id,
        },
    )

print("\u2705 task defined")

## Steps 5–6: expand cases and run the experiment

`ChaosCase.expand(...)` produces one case per effect map plus a healthy **baseline** — that's the before/after: baseline vs each failure. We score with two chaos-aware evaluators from `strands_evals.evaluators.chaos`:

- `FailureCommunicationEvaluator` — did the agent tell the user something went wrong?
- `RecoveryStrategyEvaluator` — did it take a sensible recovery action instead of guessing?

LLM behavior is stochastic, so we run the whole suite a few rounds and average — one run is a sample, not a measurement.

> ⏳ Runs the agent `ROUNDS × 4` times against the real API plus two LLM judges (~1–3 min).

In [ ]:
CONDITIONS = ["baseline", "timeout", "network", "corrupt"]
ROUNDS = 3  # average over a few rounds to see past run-to-run noise

async def run_suite():
    scores = defaultdict(list)
    for r in range(ROUNDS):
        cases = ChaosCase.expand([Case(name="trip", input=QUERY)], effect_maps, include_no_effect_baseline=True)
        experiment = ChaosExperiment(
            cases=cases,
            evaluators=[FailureCommunicationEvaluator(model=MODEL), RecoveryStrategyEvaluator(model=MODEL)],
        )
        report = await experiment.run_evaluations_async(task=weather_agent_task, max_workers=1)  # TracedHandler shares one span exporter; run sequentially so traces don't interleave
        for meta, score in zip(report.cases, report.scores):
            scores[meta["name"].split("|")[-1]].append(score)
        print(f"  round {r + 1}/{ROUNDS} done")
    return scores

scores = asyncio.run(run_suite())

print("\nCondition    mean   resilience scores per round")
for c in CONDITIONS:
    print(f"  {c:<9} {statistics.mean(scores[c]):.2f}   {[round(s, 2) for s in scores[c]]}")

In [ ]:
import matplotlib
matplotlib.rcParams["figure.facecolor"] = "white"
import matplotlib.pyplot as plt

means = [statistics.mean(scores[c]) for c in CONDITIONS]
colors = ["#BDBDBD"] + ["#FF7043"] * (len(CONDITIONS) - 1)  # baseline grey, failures coral

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(CONDITIONS, means, color=colors)
ax.set_ylabel(f"Resilience score, mean of {ROUNDS} rounds (0.0\u20131.0)")
ax.set_ylim(0, 1.0)
ax.set_title("Agent resilience: healthy baseline vs injected tool failures", fontweight="bold")
for bar, v in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

baseline = statistics.mean(scores["baseline"])
worst_score, worst_name = min((statistics.mean(scores[c]), c) for c in CONDITIONS if c != "baseline")
print(f"\u2192 baseline {baseline:.2f}; worst-handled failure: {worst_name} at {worst_score:.2f}")

## What the numbers say

On the healthy **baseline** the agent answers normally. Under injected failures its resilience scores **drop** — that gap is exactly what chaos testing surfaces, and you'd never see it on a happy-path test. Which failure hurts most varies by model and run; that's why we averaged a few rounds, and why the run-to-run spread in the table matters as much as the mean.

We changed *nothing* in the tool or the task to inject these failures — only the `ChaosPlugin` and the effect maps. That's the point of the workflow: resilience testing composes onto an existing agent.

## Key takeaways

- **Chaos testing measures resilience you can't see on the happy path.** Inject `Timeout`, `NetworkError`, or `CorruptValues` per tool and watch the score drop from baseline.
- **It's additive — no task-code changes.** `ChaosPlugin()` in `plugins` plus an effect map is the whole setup; the plugin works through Strands' tool-call hooks.
- **Score the trajectory, not just the answer.** `@eval_task(TracedHandler())` + `trace_attributes` feed the agent's traced trajectory to the chaos-aware evaluators automatically.
- **One run is a sample.** LLM behavior is stochastic; average several rounds and read the spread, or you're reporting noise.

## Frequently asked questions

**What is chaos testing for AI agents?**
Injecting controlled failures (timeouts, network errors, corrupted responses) into an agent's tool calls during evaluation, to measure how it behaves when its environment breaks instead of only testing the happy path.

**Do I have to change my agent or tools to use it?**
No. Add `ChaosPlugin()` to the agent's `plugins` and declare effect maps. The plugin injects failures through the tool-call hooks; your tool and task code are unchanged.

**Why run the suite multiple times?**
LLM responses are stochastic — the same injected failure can score differently across runs. The mean and spread over several rounds is the measurement; a single run can mislead.

**How is chaos testing different from red teaming?**
Chaos testing is bad luck (do broken tools break your agent?). Red teaming is bad intent (does the agent misbehave when a user attacks it?). See [02 - Red Teaming](../02-red-teaming/).

## References

- [Strands Evals chaos testing docs](https://strandsagents.com/docs/user-guide/evals-sdk/chaos_testing/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)
- [Strands Evals (GitHub)](https://github.com/strands-agents/evals)
- [Open-Meteo API](https://open-meteo.com) · tool adapted from [Ricardo Ceci's `curso-strands-agentcore-2026`](https://github.com/ricardoceci/curso-strands-agentcore-2026)

---

Gracias!

🇻🇪🇨🇱 [Dev.to](https://dev.to/elizabethfuentes12) · [GitHub](https://github.com/elizabethfuentes12/)